# CEG-WM Content V8 — formal initial user-only GPU handoff

This Drive-first notebook invokes the frozen initial formal runner exactly once for `content-v8-bd6269861412-0ba01f405106`, from `stage-a-content-v8-v2-spatial-lf-detector-domain-iss@bd6269861412b8628009238ace84d3278ff1e17a`. It is an operational handoff only; it does not interpret outcomes or make a scientific claim.

Create Colab Secrets named `CEG_WM_ROOT_KEY` and `HF_TOKEN`. Run once from top to bottom and stop after any failure or interruption.

## 1. Mount Drive, then prove the fresh execution checkout

Drive mounting is first. The source checkout and the exact-bound local and Drive destinations must be absent before checkout and installation.

In [ ]:
from google.colab import drive
try:
    drive.mount("/content/drive")
except BaseException:
    print('CEGWM_CONTENT_V8_FORMAL_HANDOFF_FAILURE {\"error_class\":\"OtherOperationalError\",\"execution_exact\":\"bd6269861412b8628009238ace84d3278ff1e17a\",\"run_id\":\"content-v8-bd6269861412-0ba01f405106\",\"stage\":\"drive_mount\",\"status\":\"operational_failure\"}', flush=True)
    HANDOFF_FAILED = True
else:
    HANDOFF_FAILED = False

import json
import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
BRANCH = "stage-a-content-v8-v2-spatial-lf-detector-domain-iss"
EXACT = "bd6269861412b8628009238ace84d3278ff1e17a"
PROTOCOL_DIGEST = "7670d54434906ae246ef76c097bf54f997c3a0ca6b036c0f55e0fe5b31489a1c"
RUNNER_MODULE = "experiments.run_content_v8_formal_initial"
RUN_ID = "content-v8-bd6269861412-0ba01f405106"
FAILURE_PREFIX = "CEGWM_CONTENT_V8_FORMAL_HANDOFF_FAILURE"
ARTIFACT_PREFIX = "CEGWM_CONTENT_V8_FORMAL_ARTIFACT"
repo = pathlib.Path("/content/cegwm-content-v8-formal-initial-source")
artifact_sink = pathlib.Path("/content/drive/MyDrive/CEG-WM/content_v8_v2_spatial_lf_detector_domain_iss_formal_initial")
bound_drive_run = artifact_sink / RUN_ID
RUNNER_ATTEMPTED = False

_ALLOWED_ERRORS = {"CalledProcessError", "FileExistsError", "ImportError", "ModuleNotFoundError", "OSError", "RuntimeError", "TypeError", "UnicodeDecodeError", "ValueError"}

def fail(stage, error_class="RuntimeError"):
    global HANDOFF_FAILED
    if HANDOFF_FAILED:
        return
    HANDOFF_FAILED = True
    if error_class not in _ALLOWED_ERRORS:
        error_class = "OtherOperationalError"
    line = FAILURE_PREFIX + " " + json.dumps({"status": "operational_failure", "run_id": RUN_ID, "execution_exact": EXACT, "protocol_digest": PROTOCOL_DIGEST, "stage": stage, "error_class": error_class}, sort_keys=True, separators=(",", ":"))
    if len(line.encode("utf-8")) <= 4096:
        print(line, flush=True)

def git(*args):
    return subprocess.run(["git", *args], cwd=repo, check=True, capture_output=True, text=True).stdout.strip()

if not HANDOFF_FAILED:
    try:
        if repo.exists() or bound_drive_run.exists():
            raise FileExistsError
        subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(repo)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if git("branch", "--show-current") != BRANCH or git("rev-parse", "HEAD") != EXACT or git("status", "--porcelain") != "" or bound_drive_run.exists():
            raise RuntimeError
    except BaseException as error:
        fail("source_checkout_identity_validation", type(error).__name__)


## 2. Install only the checked-out project and invoke the runner once

Both secrets enter only the child environment and are cleared promptly. The child output is bounded and never printed; the runner is the sole owner of create-only runtime and terminal artifacts.

In [ ]:
if not HANDOFF_FAILED and not RUNNER_ATTEMPTED:
    RUNNER_ATTEMPTED = True
    process = None
    runner_env = None
    root_key = ""
    hf_token = ""
    captured = bytearray()
    capture_overflow = False
    runner_rc = None
    launch_error = None
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", str(repo)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if git("branch", "--show-current") != BRANCH or git("rev-parse", "HEAD") != EXACT or git("status", "--porcelain") != "" or bound_drive_run.exists():
            raise RuntimeError
        from google.colab import userdata
        root_key = userdata.get("CEG_WM_ROOT_KEY")
        hf_token = userdata.get("HF_TOKEN")
        if not isinstance(root_key, str) or not root_key.strip() or not isinstance(hf_token, str) or not hf_token.strip():
            raise RuntimeError
        secret_markers = ("TOKEN", "KEY", "SECRET", "PASSWORD", "CREDENTIAL")
        runner_env = {name: value for name, value in os.environ.items() if not any(marker in name.upper() for marker in secret_markers)}
        runner_env["CEG_WM_ROOT_KEY"] = root_key
        runner_env["HF_TOKEN"] = hf_token
        root_key = ""
        hf_token = ""
        process = subprocess.Popen([sys.executable, "-m", RUNNER_MODULE, "--repo-root", str(repo), "--expected-exact", EXACT, "--artifact-sink", str(artifact_sink)], cwd=repo, env=runner_env, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
        while True:
            chunk = process.stdout.read(1024)
            if not chunk:
                break
            remaining = max(0, 8192 - len(captured))
            captured.extend(chunk[:remaining])
            capture_overflow = capture_overflow or len(chunk) > remaining
        runner_rc = process.wait()
    except BaseException as error:
        launch_error = type(error).__name__
        if process is not None and process.poll() is None:
            process.kill()
            process.wait()
    finally:
        root_key = ""
        hf_token = ""
        if runner_env is not None:
            runner_env.pop("CEG_WM_ROOT_KEY", None)
            runner_env.pop("HF_TOKEN", None)
        runner_env = None
    captured.clear()
    if launch_error is not None:
        fail("formal_runner_launch", launch_error)
    elif runner_rc != 0:
        fail("formal_runner_nonzero")
    elif capture_overflow:
        fail("formal_runner_stdout_overflow")


## 3. Return the existing runner-owned terminal artifact pair

This cell is runner-free, write-free, resume-free, and download-free. It reads only the existing SHA-256 sidecar and verifies its filename binding; it never reads ZIP bytes.

In [ ]:
import re
run_dir = artifact_sink / RUN_ID
prior_failure = bool(globals().get("HANDOFF_FAILED", False))
if not prior_failure:
    try:
        archive_path = run_dir / "terminal" / (RUN_ID + ".zip")
        sidecar_path = archive_path.with_name(archive_path.name + ".sha256")
        if not archive_path.is_file() or not sidecar_path.is_file():
            raise RuntimeError
        match = re.fullmatch(r"([0-9a-f]{64})  ([^\s]+)\n", sidecar_path.read_text(encoding="ascii"))
        if match is None or match.group(2) != archive_path.name:
            raise RuntimeError
        line = ARTIFACT_PREFIX + " " + json.dumps({"status": "drive_artifacts_ready", "run_id": RUN_ID, "execution_exact": EXACT, "protocol_digest": PROTOCOL_DIGEST, "artifact_kind": "terminal", "archive_path": str(archive_path), "sidecar_path": str(sidecar_path), "sha256": match.group(1)}, sort_keys=True, separators=(",", ":"))
        if len(line.encode("utf-8")) > 4096:
            raise RuntimeError
        print(line, flush=True)
    except BaseException:
        fail("artifact_pair_validation")


## Stop boundary

Return only the bounded Drive artifact receipt or a single sanitized failure line. Do not rerun, refit, resume, mutate artifacts, expose child streams or secrets, download files, or interpret scientific outcomes.